# Stage 1 — Compare and Late-Fuse Validation Predictions

This notebook uses only the `val_predictions.csv` files created by notebooks 01–03.
No dataset CSV, manifest, split module, or model weights are loaded here.

Search of fusion weights and threshold is validation-only. Do not use `test.csv` here.

## 1. Setup

In [ ]:
from __future__ import annotations

import sys
import json
from itertools import combinations
from pathlib import Path

import pandas as pd

try:
    import blackbox_detection  # noqa: F401
except ModuleNotFoundError:
    _root = Path.cwd()
    while _root != _root.parent and not (_root / "pyproject.toml").is_file():
        _root = _root.parent
    sys.path.insert(0, str(_root / "src"))

from blackbox_detection.stage1.fusion import (
    compare_models,
    fused_predictions,
    load_prediction_tables,
    prediction_correlation,
    search_late_fusion,
)
from blackbox_detection.stage1.evaluator import save_predictions

## 2. Paths

In [ ]:
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent

OUTPUT_ROOT = REPO_ROOT / "outputs" / "stage1"
FUSION_DIR = OUTPUT_ROOT / "fusion"
FUSION_DIR.mkdir(parents=True, exist_ok=True)

print("outputs:", OUTPUT_ROOT)

## 3. Available validation predictions

In [ ]:
CANDIDATES = {
    "videomaev2_b": "V1 VideoMAEv2-B",
    "vjepa2_1_b": "V2 V-JEPA 2.1-B",
    "bayar_resnet18": "F1 Bayar + ResNet18",
    "cdc": "F2 CDC",
    "chromaticity": "F3 CMA-inspired chromaticity",
    "frequency": "F4 Frequency/Moiré",
    "lcdf": "F5 LC&DF-inspired",
}

PREDICTION_PATHS = {
    key: OUTPUT_ROOT / key / "val_predictions.csv"
    for key in CANDIDATES
    if (OUTPUT_ROOT / key / "val_predictions.csv").is_file()
}

if not PREDICTION_PATHS:
    raise FileNotFoundError(
        "No val_predictions.csv found. Run notebooks 01/02/03 first."
    )

print("available:")
for key, path in PREDICTION_PATHS.items():
    print(f"  {key:18s} -> {path}")

## 4. Load / compare

In [ ]:
wide = load_prediction_tables(PREDICTION_PATHS)
print("validation videos:", len(wide))
print("class balance:", wide["label"].value_counts().to_dict())

comparison = compare_models(wide)
comparison.insert(1, "description", comparison["model"].map(CANDIDATES))
display(comparison)

In [ ]:
if len(PREDICTION_PATHS) >= 2:
    display(prediction_correlation(wide))

## 5. Late fusion

In [ ]:
available = list(PREDICTION_PATHS)
rows = []
results = {}

# Pair searches are cheap and directly show complementarity.
for pair in combinations(available, 2):
    result = search_late_fusion(wide, pair, weight_step=0.05)
    results[pair] = result
    rows.append(result.as_row())

ensemble_table = (
    pd.DataFrame(rows)
    .sort_values("macro_f1", ascending=False, kind="mergesort")
    .reset_index(drop=True)
    if rows
    else pd.DataFrame()
)
display(ensemble_table)

In [ ]:
# Optional three-model search: two video backbones + best available forensic model.
video_models = [m for m in ("videomaev2_b", "vjepa2_1_b") if m in available]
forensic_models = [
    m for m in ("lcdf", "chromaticity", "frequency", "bayar_resnet18", "cdc")
    if m in available
]

three_model_result = None
if len(video_models) == 2 and forensic_models:
    forensic_rank = (
        comparison[comparison["model"].isin(forensic_models)]
        .sort_values("macro_f1", ascending=False)
    )
    best_forensic = forensic_rank.iloc[0]["model"]
    group = (*video_models, best_forensic)
    three_model_result = search_late_fusion(wide, group, weight_step=0.05)
    print("three-model candidate:", group)
    display(pd.DataFrame([three_model_result.as_row()]))

## 6. Save best validation fusion

In [ ]:
all_results = list(results.values())
if three_model_result is not None:
    all_results.append(three_model_result)

if not all_results:
    print("Need at least two validation prediction files for fusion.")
else:
    best = max(all_results, key=lambda result: result.macro_f1)
    best_predictions = fused_predictions(wide, best)

    save_predictions(best_predictions.drop(columns=["threshold"]), FUSION_DIR / "val_predictions.csv")
    (FUSION_DIR / "fusion_summary.json").write_text(
        json.dumps(
            {
                "models": list(best.models),
                "weights": list(best.weights),
                "threshold": float(best.threshold),
                "macro_f1": float(best.macro_f1),
                "gain_over_best_single": float(best.gain_over_best_single),
                "per_class_f1": best.per_class_f1,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    print("best fusion:", best.as_row())
    print("saved to:", FUSION_DIR)

## Important

`test.csv` is not used here. Fusion weights and the decision threshold are selected on validation only.
After the architecture/weights are frozen, apply those fixed choices once to the internal test split.